# Kote Charts quickstart — Bitcoin on-chain data in a notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kote-charts/kote-charts/blob/main/examples/jupyter/quickstart.ipynb)

MVRV Z-Score across halving cycles, today's key readings and the methodology behind them, from
[Kote Charts](https://kotecharts.com) — Bitcoin on-chain analytics computed from our own full node.

- The **metric catalog** (formulas, fields, units, update cadence) needs **no key**.
- **Chart data** needs an API key — create one at [kotecharts.com/account](https://kotecharts.com/account)
  (no charge for non-commercial use). Limit: 3 requests per second per key.

**Data licence:** non-commercial use with attribution — keep *"Data: Kote Charts — kotecharts.com"* (or a link to
each chart page) next to anything you publish from this notebook. Commercial use requires prior written permission:
[kotecharts.com/license](https://kotecharts.com/license). Educational only, never financial advice.

In [ ]:
%pip install -q "kotecharts[pandas]" matplotlib

## 1 · Methodology — no key needed

In [ ]:
from kotecharts import KoteCharts

kc = KoteCharts()  # metric() and metrics() work without a key
m = kc.metric("mvrv-z-score")
print(m["title"], "—", m["formula"])
print("Updates:", m["updateCadence"])
print("Fields:", ", ".join(f'{f["key"]} ({f.get("unit")})' for f in m["fields"]))

## 2 · Your API key

In Colab, add it once under 🔑 **Secrets** as `KOTE_API_KEY`; elsewhere set the `KOTE_API_KEY` environment variable
or paste it when asked. The key is never written into the notebook.

In [ ]:
import os
from getpass import getpass

try:  # Google Colab secrets
    from google.colab import userdata  # type: ignore

    os.environ.setdefault("KOTE_API_KEY", userdata.get("KOTE_API_KEY") or "")
except Exception:
    pass
if not os.environ.get("KOTE_API_KEY"):
    os.environ["KOTE_API_KEY"] = getpass("Kote Charts API key (kote_…): ")

kc = KoteCharts()  # reads KOTE_API_KEY

## 3 · MVRV Z-Score across halving cycles

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

HALVINGS = ["2012-11-28", "2016-07-09", "2020-05-11", "2024-04-20"]
df = kc.chart("mvrv-z-score", start="2011-07-20", granularity="week").to_pandas()

ax = df["mvrvZ"].plot(figsize=(11, 4), lw=1.2, title="MVRV Z-Score (weekly)")
for h in HALVINGS:
    ax.axvline(pd.Timestamp(h, tz="UTC"), color="grey", ls="--", lw=0.8)
ax.axhline(0, color="black", lw=0.6)
ax.set_xlabel("")
ax.annotate(
    "Data: Kote Charts — kotecharts.com/c/mvrv-z-score",
    xy=(1, -0.14), xycoords="axes fraction", ha="right", fontsize=8, color="grey",
)
plt.show()

In [ ]:
edges = [df.index.min(), *(pd.Timestamp(h, tz="UTC") for h in HALVINGS), df.index.max()]
rows = []
for start, end in zip(edges[:-1], edges[1:]):
    z = df.loc[start:end, "mvrvZ"].dropna()
    if z.empty:
        continue
    rows.append({
        "cycle": f"{start.date()} → {end.date()}",
        "peak": round(z.max(), 2), "peak on": z.idxmax().date(),
        "low": round(z.min(), 2), "low on": z.idxmin().date(),
    })
pd.DataFrame(rows)

## 4 · Today's readings

`latest()` returns the newest **closed** row. Flow metrics such as SOPR and exchange netflow also publish 5-minute
buckets after every block (`granularity="5m"`, last 30 days); valuation metrics close once per UTC day.

In [ ]:
readings = []
for chart_id, field in [("mvrv-z-score", "mvrvZ"), ("nupl", "nupl"), ("sopr", "sopr"), ("exchange-netflow", "exNetBtc")]:
    row = kc.latest(chart_id) or {}
    readings.append({"chart": chart_id, "field": field, "value": row.get(field), "as of (UTC)": row.get("t")})
pd.DataFrame(readings)

## Cite the source

Every API answer carries a ready citation. Publishing a chart or a number? Link its chart page; an app, bot or
dashboard can instead show one *"Data: Kote Charts — kotecharts.com"* credit.

In [ ]:
for chart_id in ["mvrv-z-score", "nupl", "sopr", "exchange-netflow"]:
    print(kc.metric(chart_id)["cite"])

**Next:** every chart page on [kotecharts.com/charts](https://kotecharts.com/charts) has copy-paste snippets for that
chart · [Python SDK](https://github.com/kote-charts/kote-charts/tree/main/python) ·
[Google Sheets template](https://github.com/kote-charts/kote-charts/tree/main/examples/google-sheets) ·
[MCP server for AI agents](https://kotecharts.com/mcp) (no key).